<!--
---
PURPOSE: Extract trials and behavioral events; align to timebase.
REQUIRES:
  - outputs/reports/session_inventory.parquet
PRODUCES:
  - outputs/behavior/session_{id}_trials.parquet
  - outputs/behavior/session_{id}_events.parquet
WHAT_NEXT: notebooks/04_Eye_Tracking_QC_and_Features.ipynb
---
-->

# 03 Behavior and Task Alignment

**Purpose**
Extract trials and behavioral events; align to timebase.

**Requires**
- `outputs/reports/session_inventory.parquet`

**Produces**
- `outputs/behavior/session_{id}_trials.parquet`
- `outputs/behavior/session_{id}_events.parquet`

**What to run next**
- `notebooks/04_Eye_Tracking_QC_and_Features.ipynb`

We extract trials and behavioral events and generate QC summaries.


## Environment Setup
We add the repo `src/` to the Python path so notebooks can import shared modules.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

## Prerequisite Check
We parse the notebook header and validate required artifacts before running downstream steps.

In [ ]:
from pathlib import Path
from reports import parse_notebook_header, validate_prerequisites

nb_path = Path.cwd() / "notebooks" / "03_Behavior_and_Task_Alignment.ipynb"
header = parse_notebook_header(nb_path)
missing = validate_prerequisites(header.get("REQUIRES", []))
if missing:
    print("Missing prerequisites:")
    for item in missing:
        print(" -", item)
else:
    print("All prerequisites satisfied.")

## Step 1: Load trials and events
This uses NWB as the canonical source.

In [ ]:
from io_sessions import load_sessions_csv, get_session_bundle
from features_task import derive_task_features
from timebase import write_parquet_with_timebase
from config import get_config, make_provenance
from viz import plot_behavior_summary
import pandas as pd

cfg = get_config()
sessions = load_sessions_csv()
SESSION_IDS = sessions.session_id.tolist()[:1]

for session_id in SESSION_IDS:
    bundle = get_session_bundle(session_id, sessions)
    trials, events = bundle.load_trials_and_events()
    plot_behavior_summary(trials)
    task_features = derive_task_features(trials, events)
    if task_features is not None:
        out_path = cfg.outputs_dir / "behavior" / f"session_{session_id}_task_features.parquet"
        write_parquet_with_timebase(
            task_features,
            out_path,
            provenance=make_provenance(session_id, "nwb"),
            required_columns=["t"],
        )